# Hybrid Search Evaluation with Dense and Sparse Embeddings

This notebook evaluates retrieval performance using:
- Dense embeddings from a fine-tuned model
- Sparse embeddings (BM25)
- Hybrid approach combining both methods

We'll evaluate using multiple metrics: precision@10, recall@10, mrr@10, dcg@10 on two datasets:
1. SciFact dataset
2. Synthetic software jobs dataset

## Import Data Sets Needed
 - Data set for questions = queries
 - Data set for answers = text
 - Data set for QAR = qrels

In [4]:
from datasets import load_dataset

dataset_answers = load_dataset("BeIR/scifact", "corpus", split="corpus" , trust_remote_code=True)
dataset_answers[0]

{'_id': '4983',
 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.',
 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, t

In [5]:
dataset_questions = load_dataset("BeIR/scifact", "queries", split="queries" , trust_remote_code=True)
dataset_questions[0]

{'_id': '0',
 'title': '',
 'text': '0-dimensional biomaterials lack inductive properties.'}

In [6]:
qrels = load_dataset("BeIR/scifact-qrels", split="train" , trust_remote_code=True)
qrels[0]

{'query-id': 0, 'corpus-id': 31715818, 'score': 1}

In [5]:
from collections import defaultdict

class CustomQrels:
    def __init__(self, qrels_dict, name=None):
        self.qrels = qrels_dict
        self.name = name
        
    def get_query_ids(self):
        """Return all query IDs in the qrels."""
        return list(self.qrels.keys())
    
    def get_document_ids(self, query_id):
        """Return all document IDs for a given query ID."""
        return list(self.qrels.get(query_id, {}).keys())
    
    def get_relevance(self, query_id, doc_id):
        """Return the relevance score for a query-document pair."""
        return self.qrels.get(query_id, {}).get(doc_id, 0)
    
    def __str__(self):
        return f"CustomQrels(name={self.name}, queries={len(self.get_query_ids())})"

## Transform qrels to dictionary format
Converting relevance judgments (qrels) to nested dictionary format for use in evaluation


In [9]:
from collections import defaultdict

qrels_dict = defaultdict(dict)
if not isinstance(qrels, CustomQrels):
    for entry in qrels:
        qrels_id = str(entry['query-id'])
        doc_id = str(entry['corpus-id'])
        qrels_dict[qrels_id][doc_id] = entry['score']
custom_qrels = CustomQrels(qrels_dict , name = "scifact")

# Setting up Qdrant vector database
Initialize Qdrant client and create collection for dense, sparse and late-interaction embeddings

In [10]:
from qdrant_client import QdrantClient
client = QdrantClient(":memory:" , timeout = 600)


In [ ]:
from fastembed import TextEmbedding
from fastembed.late_interaction import LateInteractionTextEmbedding
from fastembed.sparse.bm25 import Bm25
from qdrant_client import models

dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
late_interaction = LateInteractionTextEmbedding("colbert-ir/colbertv2.0" , device = "cuda")
bm25_embedding_model = Bm25("Qdrant/bm25")

sample_text = ["This is a sample document"]
dense_embedding = list(dense_model.passage_embed(sample_text))[0]
late_embedding = list(late_interaction.passage_embed(sample_text))[0]

# Create collection with vector configurations for all embedding types
client.create_collection(
    "scifact",
    vectors_config ={
        "all-MiniLM-L6-v2" : models.VectorParams(
            size = len(dense_embedding),
            distance = models.Distance.COSINE
        ),
        "colbertv2.0" : models.VectorParams(
            size = len(late_embedding[0]),
            distance = models.Distance.COSINE,
            multivector_config = models.MultiVectorConfig(
                comparator = models.MultiVectorComparator.MAX_SIM,
            ),
        ),
    },
    sparse_vectors_config = {
        "bm25" : models.SparseVectorParams(
            modifier= models.Modifier.IDF,
        )
    }
)

# Loading dataset into vector database
Process and upload dataset to Qdrant with dense, late-interaction and sparse embeddings

In [14]:
import tqdm
batch_size = 4

for batch in tqdm.tqdm(dataset_answers.iter(batch_size=batch_size) , total = len(dataset_answers) // batch_size):
    dense_embeddings = list(dense_model.passage_embed(batch["text"]))
    late_embeddings = list(late_interaction.passage_embed(batch["text"]))
    sparse_embeddings = list(bm25_embedding_model.passage_embed(batch["text"]))

    client.upload_points(
        "scifact",
        points = [
            models.PointStruct(
                id = int(batch["_id"][i]),
                vector = {
                    "all-MiniLM-L6-v2": dense_embeddings[i].tolist(),
                    "colbertv2.0": late_embeddings[i].tolist(),
                    "bm25": models.SparseVector(
                        indices = sparse_embeddings[i].as_object()["indices"].tolist(),
                        values = sparse_embeddings[i].as_object()["values"].tolist(),
                    ),
                },
                payload = {
                    "_id": batch["_id"][i],
                    "title": batch["title"][i],
                    "text": batch["text"][i],
                }
            )
            for i , _ in enumerate(batch["_id"])
        ],
        batch_size=batch_size,
    )

1296it [15:34,  1.39it/s]                          


# Creating query embeddings
Generate dense, sparse and late-interaction embeddings for search queries

In [15]:
import tqdm
import numpy as np
import gc

# Initialize empty lists for embeddings
dense_vectors = []
sparse_vectors = []
late_vectors = []

# Process in batches using the proper dataset method
BATCH_SIZE = 16  # You can adjust this value

# Use the built-in iter method of HuggingFace datasets
for batch in tqdm.tqdm(dataset_questions.iter(batch_size=BATCH_SIZE), total=len(dataset_questions)//BATCH_SIZE):
    # batch["text"] is already a list of texts
    batch_texts = batch["text"]
    
    # Process in batch
    batch_dense_embeddings = list(dense_model.query_embed(batch_texts))
    batch_late_embeddings = list(late_interaction.query_embed(batch_texts))
    batch_sparse_embeddings = list(bm25_embedding_model.query_embed(batch_texts))
    
    # Add to main lists
    dense_vectors.extend(batch_dense_embeddings)
    late_vectors.extend(batch_late_embeddings)
    sparse_vectors.extend(batch_sparse_embeddings)
    
    # Optional: Clear some memory
    gc.collect()

70it [00:30,  2.29it/s]                        


# Evaluation metrics implementation
Custom implementation of precision@k, recall@k, mrr@k, dcg@k metrics

In [6]:
class SimpleRun:
    def __init__(self, run_dict, name=None):
        self.run_dict = run_dict
        self.name = name
    
    def __str__(self):
        return f"SimpleRun(name={self.name}, queries={len(self.run_dict)})"

# Define evaluation metrics
def precision_at_k(qrels_dict, run_dict, k=10):
    """Calculate precision@k for all queries"""
    precisions = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
            
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        top_k_doc_ids = [doc_id for doc_id, _ in top_k_docs]
        
        # Count relevant documents in top k
        relevant_docs = sum(1 for doc_id in top_k_doc_ids 
                           if doc_id in qrels_dict[query_id] and qrels_dict[query_id][doc_id] > 0)
        
        # Calculate precision
        precision = relevant_docs / k if k > 0 else 0
        precisions.append(precision)
    
    # Return average precision across all queries
    return sum(precisions) / len(precisions) if precisions else 0

def mrr_at_k(qrels_dict, run_dict, k=10):
    """Calculate Mean Reciprocal Rank@k for all queries"""
    reciprocal_ranks = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
            
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        
        # Find first relevant document
        reciprocal_rank = 0
        for rank, (doc_id, _) in enumerate(top_k_docs, start=1):
            if doc_id in qrels_dict[query_id] and qrels_dict[query_id][doc_id] > 0:
                reciprocal_rank = 1.0 / rank
                break
                
        reciprocal_ranks.append(reciprocal_rank)
    
    # Return average MRR across all queries
    return sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0

def recall_at_k(qrels_dict, run_dict, k=10):
    """Calculate recall@k for all queries"""
    recalls = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
        
        # Get total number of relevant documents for this query
        total_relevant = sum(1 for doc_id, score in qrels_dict[query_id].items() if score > 0)
        
        if total_relevant == 0:
            continue  # Skip queries with no relevant documents
        
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        top_k_doc_ids = [doc_id for doc_id, _ in top_k_docs]
        
        # Count relevant documents in top k
        relevant_retrieved = sum(1 for doc_id in top_k_doc_ids 
                              if doc_id in qrels_dict[query_id] and qrels_dict[query_id][doc_id] > 0)
        
        # Calculate recall
        recall = relevant_retrieved / total_relevant
        recalls.append(recall)
    
    # Return average recall across all queries
    return sum(recalls) / len(recalls) if recalls else 0

def dcg_at_k(qrels_dict, run_dict, k=10):
    """Calculate DCG@k for all queries"""
    dcg_scores = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
            
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        
        # Calculate DCG
        dcg = 0
        for i, (doc_id, _) in enumerate(top_k_docs, start=1):
            rel = qrels_dict[query_id].get(doc_id, 0)
            # Using 2^rel - 1 for graded relevance
            dcg += (2**rel - 1) / np.log2(i + 1)
            
        dcg_scores.append(dcg)
    
    # Return average DCG across all queries
    return sum(dcg_scores) / len(dcg_scores) if dcg_scores else 0

def simple_evaluate(qrels_dict, run_dict, metrics=None):
    """Evaluate a run against qrels using specified metrics"""
    if metrics is None:
        metrics = ["precision@10", "recall@10", "mrr@10", "dcg@10"]
    
    results = {}
    
    for metric in metrics:
        if '@' in metric:
            name, k_str = metric.split('@')
            try:
                k = int(k_str)
                if name == "precision":
                    results[metric] = precision_at_k(qrels_dict, run_dict, k=k)
                elif name == "recall":
                    results[metric] = recall_at_k(qrels_dict, run_dict, k=k)
                elif name == "mrr":
                    results[metric] = mrr_at_k(qrels_dict, run_dict, k=k)
                elif name == "dcg":
                    results[metric] = dcg_at_k(qrels_dict, run_dict, k=k)
            except ValueError:
                # If k is not a valid integer, skip this metric
                pass
    
    return results

# Evaluate Individual Embedding Models
Testing and comparing the performance of different embedding approaches:
- Dense vectors (sentence-transformers model)
- Sparse vectors (BM25)
- Hybrid approach (combination of dense and sparse)

In [25]:
def hybrid_search(
    client, 
    collection_name, 
    query_id,
    dense_vector, 
    sparse_vector,
    dense_weight=0.5, 
    sparse_weight=0.5,
    limit=10, 
    with_payload=False
):
    # Normalize weights to sum to 1.0
    total = dense_weight + sparse_weight
    dense_weight /= total
    sparse_weight /= total

    # Set search limit higher to ensure good coverage
    search_limit = limit * 3

    # Query with dense vector
    dense_results = client.query_points(
        collection_name,
        query=dense_vector,
        using="all-MiniLM-L6-v2",
        with_payload=with_payload,
        limit=search_limit,
    )

    # Query with sparse vector
    sparse_results = client.query_points(
        collection_name,
        query=models.SparseVector(
            indices=sparse_vector.as_object()["indices"].tolist(),
            values=sparse_vector.as_object()["values"].tolist(),
        ),
        using="bm25",
        with_payload=with_payload,
        limit=search_limit,
    )

    # Initialize combined scores dictionary
    combined_scores = {}

    # Process dense results with normalization
    dense_scores = {str(point.id): point.score for point in dense_results.points}
    if dense_scores:
        max_dense = max(dense_scores.values())
        min_dense = min(dense_scores.values())
        range_dense = max_dense - min_dense if max_dense > min_dense else 1.0

        for doc_id, score in dense_scores.items():
            # Normalize to 0-1 range
            norm_score = (score - min_dense) / range_dense if range_dense > 0 else score
            combined_scores[doc_id] = dense_weight * norm_score

    # Process sparse results with normalization
    sparse_scores = {str(point.id): point.score for point in sparse_results.points}
    if sparse_scores:
        max_sparse = max(sparse_scores.values())
        min_sparse = min(sparse_scores.values())
        range_sparse = max_sparse - min_sparse if max_sparse > min_sparse else 1.0

        for doc_id, score in sparse_scores.items():
            # Normalize to 0-1 range
            norm_score = (score - min_sparse) / range_sparse if range_sparse > 0 else score
            if doc_id in combined_scores:
                combined_scores[doc_id] += sparse_weight * norm_score
            else:
                combined_scores[doc_id] = sparse_weight * norm_score

    # Return combined scores (not sorted or limited here to match your format)
    return combined_scores

In [27]:
# Evaluate all 3 methods: dense, sparse, and hybrid

results_summary = {}

# Dense only
run_dic = {}
for idx, query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_dense_vector = dense_vectors[idx]
    results = client.query_points(
        "scifact",
        query=question_dense_vector,
        using="all-MiniLM-L6-v2",
        with_payload=False,
        limit=10,
    )
    run_dic[query_id] = {str(point.id): point.score for point in results.points}
dense_run = SimpleRun(run_dic, name="dense_run")
dense_eval = simple_evaluate(custom_qrels.qrels, run_dic, metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"])
results_summary["dense"] = dense_eval
print("Dense:", dense_eval)

# Sparse only
run_dic = {}
for idx, query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_sparse_vector = sparse_vectors[idx]
    results = client.query_points(
        "scifact",
        query=models.SparseVector(
            indices=question_sparse_vector.as_object()["indices"].tolist(),
            values=question_sparse_vector.as_object()["values"].tolist()
        ),
        using="bm25",
        with_payload=False,
        limit=10,
    )
    run_dic[query_id] = {str(point.id): point.score for point in results.points}
sparse_run = SimpleRun(run_dic, name="sparse_run")
sparse_eval = simple_evaluate(custom_qrels.qrels, run_dic, metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"])
results_summary["sparse"] = sparse_eval
print("Sparse:", sparse_eval)

# Hybrid (dense + sparse)
run_dic = {}
for idx, query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_dense_vector = dense_vectors[idx]
    question_sparse_vector = sparse_vectors[idx]
    results = hybrid_search(
        client=client,
        collection_name="scifact",
        query_id=query_id,
        dense_vector=question_dense_vector,
        sparse_vector=question_sparse_vector,
        dense_weight=0.4,
        sparse_weight=0.6,
        limit=10,
        with_payload=False
    )
    run_dic[query_id] = results
hybrid_run = SimpleRun(run_dic, name="hybrid_run")
hybrid_eval = simple_evaluate(custom_qrels.qrels, run_dic, metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"])
results_summary["hybrid"] = hybrid_eval
print("Hybrid:", hybrid_eval)

# Optional: print all together
print("\nSummary of all methods:")
for method, scores in results_summary.items():
    print(f"{method}: {scores}")

Dense: {'precision@10': 0.08529048207663782, 'recall@10': 0.750082406262876, 'mrr@10': 0.5842746286813036, 'dcg@10': np.float64(0.664991632990785)}
Sparse: {'precision@10': 0.0892459826946848, 'recall@10': 0.804058508446642, 'mrr@10': 0.6569917789942512, 'dcg@10': np.float64(0.7290243912905734)}
Hybrid: {'precision@10': 0.09468479604449939, 'recall@10': 0.8398022249690976, 'mrr@10': 0.6834598858084644, 'dcg@10': np.float64(0.7660626221457494)}

Summary of all methods:
dense: {'precision@10': 0.08529048207663782, 'recall@10': 0.750082406262876, 'mrr@10': 0.5842746286813036, 'dcg@10': np.float64(0.664991632990785)}
sparse: {'precision@10': 0.0892459826946848, 'recall@10': 0.804058508446642, 'mrr@10': 0.6569917789942512, 'dcg@10': np.float64(0.7290243912905734)}
hybrid: {'precision@10': 0.09468479604449939, 'recall@10': 0.8398022249690976, 'mrr@10': 0.6834598858084644, 'dcg@10': np.float64(0.7660626221457494)}


In [1]:
import json
import random
from collections import defaultdict

def create_software_jobs_dataset(num_jobs=100, num_queries=25):
    # Define components for generating realistic job postings
    roles = ["Software Engineer", "Data Scientist", "DevOps Engineer", "Frontend Developer", 
             "Backend Developer", "Full Stack Developer", "ML Engineer", "Data Engineer", 
             "Cloud Architect", "Mobile Developer", "QA Engineer", "Security Engineer"]
    
    seniority = ["Junior", "Mid-level", "Senior", "Lead", "Principal", "Entry-level"]
    
    languages = ["Python", "JavaScript", "Java", "C#", "C++", "Go", "Rust", "TypeScript", 
                 "PHP", "Ruby", "Swift", "Kotlin"]
    
    frameworks = ["React", "Angular", "Vue.js", "Django", "Flask", "Spring Boot", "Express", 
                  "TensorFlow", "PyTorch", "FastAPI", ".NET", "Laravel"]
    
    databases = ["PostgreSQL", "MySQL", "MongoDB", "Redis", "Elasticsearch", "SQL Server", 
                 "Oracle", "DynamoDB", "Cassandra", "Firebase"]
    
    cloud = ["AWS", "Azure", "Google Cloud", "Kubernetes", "Docker", "Terraform", 
             "Jenkins", "GitHub Actions", "CircleCI", "ArgoCD"]
    
    locations = ["Remote", "New York", "San Francisco", "Seattle", "Austin", "Chicago", 
                "London", "Berlin", "Toronto", "Sydney", "Hybrid"]
    
    industries = ["Fintech", "Healthcare", "E-commerce", "SaaS", "Enterprise", "EdTech", 
                 "Gaming", "Cybersecurity", "AI Research", "Social Media"]
    
    # Generate job corpus
    job_corpus = []
    for i in range(1, num_jobs + 1):
        role = random.choice(roles)
        sr = random.choice(seniority)
        primary_lang = random.sample(languages, 1)[0]
        secondary_langs = random.sample([l for l in languages if l != primary_lang], 
                                      k=random.randint(0, 3))
        
        # Select random frameworks that might match the languages
        job_frameworks = random.sample(frameworks, k=random.randint(1, 4))
        job_dbs = random.sample(databases, k=random.randint(1, 3))
        job_cloud = random.sample(cloud, k=random.randint(0, 4))
        location = random.choice(locations)
        industry = random.choice(industries)
        
        skills = [primary_lang] + secondary_langs + job_frameworks + job_dbs + job_cloud
        random.shuffle(skills)
        
        # Create job description
        description = f"{sr} {role} position"
        if random.random() > 0.5:
            description += f" at a {industry} company"
        if location != "Remote":
            description += f" in {location}"
        else:
            description += " (Remote)"
        
        description += f". We're looking for someone with strong {primary_lang} skills"
        
        if secondary_langs:
            description += f" and experience with {', '.join(secondary_langs)}"
        
        description += f". The ideal candidate will have worked with {', '.join(job_frameworks[:2])} "
        description += f"and {', '.join(job_dbs)} for data storage. "
        
        if job_cloud:
            description += f"Experience with {', '.join(job_cloud)} is highly desired. "
        
        description += f"This role involves developing and maintaining applications for the {industry} sector."
        
        # Create the job posting
        job = {
            "_id": str(i),
            "title": f"{sr} {role}",
            "text": description,
            "primary_language": primary_lang,
            "all_skills": skills,
            "location": location,
            "seniority": sr,
            "industry": industry
        }
        
        job_corpus.append(job)
    
    queries = []
    for i in range(1, num_queries + 1):
        query_type = random.choice(["role", "skill", "role+skill", "role+location", 
                                   "role+skill+location", "skill+industry"])
        
        if "role" in query_type:
            role = random.choice(roles)
            if random.random() > 0.7:  # Sometimes include seniority
                role = f"{random.choice(seniority)} {role}"
        
        if "skill" in query_type:
            skill_type = random.choice(["language", "framework", "cloud", "database"])
            if skill_type == "language":
                skill = random.choice(languages)
            elif skill_type == "framework":
                skill = random.choice(frameworks)
            elif skill_type == "cloud":
                skill = random.choice(cloud)
            else:
                skill = random.choice(databases)
        
        if "location" in query_type:
            location = random.choice(locations)
        
        if "industry" in query_type:
            industry = random.choice(industries)
        
        query_text = ""
        if query_type == "role":
            query_text = f"{role} jobs"
        elif query_type == "skill":
            query_text = f"{skill} developer"
        elif query_type == "role+skill":
            query_text = f"{role} with {skill} experience"
        elif query_type == "role+location":
            query_text = f"{role} in {location}"
        elif query_type == "role+skill+location":
            query_text = f"{role} with {skill} in {location}"
        elif query_type == "skill+industry":
            query_text = f"{skill} developer for {industry}"
        
        queries.append({
            "_id": f"q{i}",
            "text": query_text
        })
    
    qrels_dict = defaultdict(dict)
    
    for query in queries:
        query_id = query["_id"]
        query_text = query["text"].lower()
        
        query_terms = query_text.split()
        
        # Find relevant jobs based on query terms
        for job in job_corpus:
            job_id = job["_id"]
            relevance_score = 0
            
            # Simple relevance scoring based on term matching
            job_text = (job["title"] + " " + job["text"]).lower()
            
            # Check for exact role match (highest relevance)
            if any(role.lower() in query_text for role in roles) and \
               any(role.lower() in job_text for role in roles if role.lower() in query_text):
                relevance_score += 2
            
            # Check for skills match
            if any(skill.lower() in query_text for skill in languages + frameworks + databases + cloud) and \
               any(skill.lower() in job_text for skill in languages + frameworks + databases + cloud 
                   if skill.lower() in query_text):
                relevance_score += 1
            
            # Check for location match
            if any(loc.lower() in query_text for loc in locations) and \
               any(loc.lower() == job["location"].lower() for loc in locations if loc.lower() in query_text):
                relevance_score += 1
            
            # Check for seniority match
            if any(sr.lower() in query_text for sr in seniority) and \
               any(sr.lower() == job["seniority"].lower() for sr in seniority if sr.lower() in query_text):
                relevance_score += 1
                
            # Check for industry match
            if any(ind.lower() in query_text for ind in industries) and \
               job["industry"].lower() in query_text:
                relevance_score += 1
                
            # Only include if somewhat relevant
            if relevance_score > 0:
                # Normalize to 0-2 range for binary or graded relevance
                final_score = min(2, relevance_score)
                qrels_dict[query_id][job_id] = final_score
    
    # Ensure each query has at least some relevant documents
    for query in queries:
        query_id = query["_id"]
        if len(qrels_dict[query_id]) < 2:
            # Add a few random jobs with low relevance
            for _ in range(2):
                random_job = random.choice(job_corpus)
                qrels_dict[query_id][random_job["_id"]] = 1
    
    return job_corpus, queries, qrels_dict

# Save the dataset to disk
job_corpus, queries, qrels_dict = create_software_jobs_dataset()

# Save as JSON files
with open("software_jobs_corpus.json", "w") as f:
    json.dump(job_corpus, f, indent=2)

with open("software_jobs_queries.json", "w") as f:
    json.dump(queries, f, indent=2)

with open("software_jobs_qrels.json", "w") as f:
    json.dump({k: dict(v) for k, v in qrels_dict.items()}, f, indent=2)

# Synthetic Dataset Generation
Creating a software jobs dataset for testing hybrid search performance on a domain-specific use case

In [15]:
import json
from collections import defaultdict
from qdrant_client import QdrantClient
from qdrant_client import models
import numpy as np

# Load your custom dataset
with open("software_jobs_corpus.json", "r") as f:
    job_corpus = json.load(f)

with open("software_jobs_queries.json", "r") as f:
    job_queries = json.load(f)

with open("software_jobs_qrels.json", "r") as f:
    job_qrels_raw = json.load(f)
    
# Convert to the format your evaluation expects
job_qrels_dict = defaultdict(dict)
for query_id, docs in job_qrels_raw.items():
    for doc_id, score in docs.items():
        job_qrels_dict[query_id][doc_id] = score

custom_qrels = CustomQrels(job_qrels_dict, name="software_jobs")

# Initialize Qdrant client
client = QdrantClient(":memory:", timeout=600)

# Use your fine-tuned model
from sentence_transformers import SentenceTransformer
from fastembed.sparse.bm25 import Bm25

job_model = SentenceTransformer("E:/vs codes/REC/job_embeddings_model_evaluated2")
bm25_embedding_model = Bm25("Qdrant/bm25")

# Create separate collections for dense and sparse vectors
# First, create a dense collection for your fine-tuned model
sample_text = ["Sample job posting text"]
sample_embedding = list(job_model.encode(sample_text))[0]

client.create_collection(
    "software_jobs_dense",
    vectors_config={
        "job_embeddings": models.VectorParams(
            size=len(sample_embedding),
            distance=models.Distance.COSINE
        )
    }
)

# Then, create a sparse collection for BM25
client.create_collection(
    "software_jobs_sparse",
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

# Upload job embeddings to both collections
for job in job_corpus:
    job_id = int(job["_id"])
    job_text = job["text"]
    
    # Upload dense embeddings to the dense collection
    job_embedding = list(job_model.encode([job_text]))[0]
    client.upload_points(
        "software_jobs_dense",
        points=[
            models.PointStruct(
                id=job_id,
                vector={"job_embeddings": job_embedding.tolist()},
                payload={
                    "_id": job["_id"],
                    "title": job["title"],
                    "text": job_text
                }
            )
        ]
    )
    
    # Upload sparse embeddings to the sparse collection
    sparse_embedding = list(bm25_embedding_model.passage_embed([job_text]))[0]
    try:
        # Use models.PointStruct for sparse vectors as well
        client.upload_points(
            "software_jobs_sparse",
            points=[
                models.PointStruct(
                    id=job_id,
                    vector={},  # Empty vector field is required
                    payload={"_id": job["_id"]},
                    sparse_vectors={
                        "bm25": {
                            "indices": sparse_embedding.as_object()["indices"].tolist(),
                            "values": sparse_embedding.as_object()["values"].tolist(),
                        }
                    }
                )
            ]
        )
    except Exception as e:
        print(f"Error uploading sparse vector: {e}")
        try:
            # Alternative approach if the first one fails
            client.upload_points(
                "software_jobs_sparse",
                points=[
                    models.PointStruct(
                        id=job_id,
                        payload={"_id": job["_id"]},
                        sparse_vectors={
                            "bm25": sparse_embedding.as_object()
                        }
                    )
                ]
            )
        except Exception as e2:
            print(f"Alternative method also failed: {e2}")

# Create query embeddings for both models
dense_query_vectors = []
sparse_query_vectors = []

for query in job_queries:
    # Create dense embedding with your fine-tuned model
    dense_query_embedding = list(job_model.encode([query["text"]]))[0]
    dense_query_vectors.append(dense_query_embedding)
    
    # Create sparse embedding with BM25
    sparse_query_embedding = list(bm25_embedding_model.query_embed([query["text"]]))[0]
    sparse_query_vectors.append(sparse_query_embedding)

# Function to perform hybrid search
def job_hybrid_search(
    client, 
    query_id,
    dense_vector, 
    sparse_vector,
    dense_weight=0.7, 
    sparse_weight=0.3,
    limit=10
):
    # Normalize weights
    total = dense_weight + sparse_weight
    dense_weight /= total
    sparse_weight /= total
    search_limit = limit * 3
    
    # Query dense collection
    dense_results = client.query_points(
        "software_jobs_dense",
        query=dense_vector,
        using="job_embeddings",
        with_payload=False,
        limit=search_limit,
    )
    
    # Query sparse collection using the correct syntax
    sparse_results = client.query_points(
        "software_jobs_sparse",
        query=models.SparseVector(
            indices=sparse_vector.as_object()["indices"].tolist(),
            values=sparse_vector.as_object()["values"].tolist(),
        ),
        using="bm25",  # The name of your sparse vector field
        with_payload=False,
        limit=search_limit,
    )
    
    # Combine and normalize scores
    combined_scores = {}
    
    # Process dense results
    dense_scores = {str(point.id): point.score for point in dense_results.points}
    if dense_scores:
        max_dense = max(dense_scores.values()) if dense_scores else 0
        min_dense = min(dense_scores.values()) if dense_scores else 0
        range_dense = max_dense - min_dense if max_dense > min_dense else 1.0
        
        for doc_id, score in dense_scores.items():
            norm_score = (score - min_dense) / range_dense if range_dense > 0 else score
            combined_scores[doc_id] = dense_weight * norm_score
    
    # Process sparse results
    sparse_scores = {str(point.id): point.score for point in sparse_results.points}
    if sparse_scores:
        max_sparse = max(sparse_scores.values()) if sparse_scores else 0
        min_sparse = min(sparse_scores.values()) if sparse_scores else 0
        range_sparse = max_sparse - min_sparse if max_sparse > min_sparse else 1.0
        
        for doc_id, score in sparse_scores.items():
            norm_score = (score - min_sparse) / range_sparse if range_sparse > 0 else score
            if doc_id in combined_scores:
                combined_scores[doc_id] += sparse_weight * norm_score
            else:
                combined_scores[doc_id] = sparse_weight * norm_score
    
    # Sort by score and limit results
    sorted_scores = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:limit]
    return dict(sorted_scores)

# Evaluate dense-only approach
dense_run_dic = {}
for idx, query in enumerate(job_queries):
    query_id = query["_id"]
    question_vector = dense_query_vectors[idx]
    
    results = client.query_points(
        "software_jobs_dense",
        query=question_vector,
        using="job_embeddings",
        with_payload=False,
        limit=10,
    )
    
    dense_run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }

# Evaluate sparse-only approach
sparse_run_dic = {}
for idx, query in enumerate(job_queries):
    query_id = query["_id"]
    question_sparse_vector = sparse_query_vectors[idx]
    
    # Use the correct query_points format for sparse vectors
    results = client.query_points(
        "software_jobs_sparse",
        query=models.SparseVector(
            indices=question_sparse_vector.as_object()["indices"].tolist(),
            values=question_sparse_vector.as_object()["values"].tolist(),
        ),
        using="bm25",  # The name of your sparse vector field
        with_payload=False,
        limit=10,
    )
    
    sparse_run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }

# Evaluate hybrid approach
hybrid_run_dic = {}
for idx, query in enumerate(job_queries):
    query_id = query["_id"]
    question_dense_vector = dense_query_vectors[idx]
    question_sparse_vector = sparse_query_vectors[idx]
    
    results = job_hybrid_search(
        client=client,
        query_id=query_id,
        dense_vector=question_dense_vector,
        sparse_vector=question_sparse_vector,
        dense_weight=0.7,  # Adjust weights as needed
        sparse_weight=0.3,
        limit=10
    )
    
    hybrid_run_dic[query_id] = results

# Evaluate all approaches
dense_eval = simple_evaluate(
    custom_qrels.qrels,
    dense_run_dic,
    metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
)

sparse_eval = simple_evaluate(
    custom_qrels.qrels,
    sparse_run_dic,
    metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
)

hybrid_eval = simple_evaluate(
    custom_qrels.qrels,
    hybrid_run_dic,
    metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
)

# Print results
print("Fine-tuned model results:", dense_eval)
print("Sparse BM25 results:", sparse_eval)
print("Hybrid search results:", hybrid_eval)

# Try different weight combinations
weight_combinations = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (0.9, 0.1)]
weight_results = {}

for dense_weight, sparse_weight in weight_combinations:
    hybrid_run = {}
    for idx, query in enumerate(job_queries):
        query_id = query["_id"]
        results = job_hybrid_search(
            client=client,
            query_id=query_id,
            dense_vector=dense_query_vectors[idx],
            sparse_vector=sparse_query_vectors[idx],
            dense_weight=dense_weight,
            sparse_weight=sparse_weight,
            limit=10
        )
        hybrid_run[query_id] = results
    
    weight_eval = simple_evaluate(
        custom_qrels.qrels,
        hybrid_run,
        metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
    )
    weight_results[f"Dense {dense_weight:.1f}, Sparse {sparse_weight:.1f}"] = weight_eval
    
print("\nHybrid results with different weights:")
for weights, results in weight_results.items():
    print(f"{weights}: {results}")

Error uploading sparse vector: 1 validation error for PointStruct
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'indices': [413...8, 1.5233973492020558]}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
Alternative method also failed: 2 validation errors for PointStruct
vector
  Field required [type=missing, input_value={'id': 1, 'payload': {'_i...4657082, 1961226291])}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'values': array...34657082, 1961226291])}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
Error uploading sparse vector: 1 validation error for PointStruct
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'indices': [169...16, 1.549807

In [ ]:
import json
from collections import defaultdict
from qdrant_client import QdrantClient
from qdrant_client import models
import numpy as np

# Load your custom dataset
with open("software_jobs_corpus.json", "r") as f:
    job_corpus = json.load(f)

with open("software_jobs_queries.json", "r") as f:
    job_queries = json.load(f)

with open("software_jobs_qrels.json", "r") as f:
    job_qrels_raw = json.load(f)
    
# Convert to the format your evaluation expects
job_qrels_dict = defaultdict(dict)
for query_id, docs in job_qrels_raw.items():
    for doc_id, score in docs.items():
        job_qrels_dict[query_id][doc_id] = score

custom_qrels = CustomQrels(job_qrels_dict, name="software_jobs")

# Initialize Qdrant client
client = QdrantClient(":memory:", timeout=600)

# Use your fine-tuned model
from sentence_transformers import SentenceTransformer
from fastembed.sparse.bm25 import Bm25

job_model = SentenceTransformer("E:/vs codes/REC/job_embeddings_model_evaluated2")
bm25_embedding_model = Bm25("Qdrant/bm25")

# Create separate collections for dense and sparse vectors
# First, create a dense collection for your fine-tuned model
sample_text = ["Sample job posting text"]
sample_embedding = list(job_model.encode(sample_text))[0]

client.create_collection(
    "software_jobs_dense",
    vectors_config={
        "job_embeddings": models.VectorParams(
            size=len(sample_embedding),
            distance=models.Distance.COSINE
        )
    }
)

# Then, create a sparse collection for BM25
client.create_collection(
    "software_jobs_sparse",
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

# Upload job embeddings to both collections
for job in job_corpus:
    job_id = int(job["_id"])
    job_text = job["text"]
    
    # Upload dense embeddings to the dense collection
    job_embedding = list(job_model.encode([job_text]))[0]
    client.upload_points(
        "software_jobs_dense",
        points=[
            models.PointStruct(
                id=job_id,
                vector={"job_embeddings": job_embedding.tolist()},
                payload={
                    "_id": job["_id"],
                    "title": job["title"],
                    "text": job_text
                }
            )
        ]
    )
    
    # Upload sparse embeddings to the sparse collection
    sparse_embedding = list(bm25_embedding_model.passage_embed([job_text]))[0]
    try:
        # Use models.PointStruct for sparse vectors as well
        client.upload_points(
            "software_jobs_sparse",
            points=[
                models.PointStruct(
                    id=job_id,
                    vector={},  # Empty vector field is required
                    payload={"_id": job["_id"]},
                    sparse_vectors={
                        "bm25": {
                            "indices": sparse_embedding.as_object()["indices"].tolist(),
                            "values": sparse_embedding.as_object()["values"].tolist(),
                        }
                    }
                )
            ]
        )
    except Exception as e:
        print(f"Error uploading sparse vector: {e}")
        try:
            # Alternative approach if the first one fails
            client.upload_points(
                "software_jobs_sparse",
                points=[
                    models.PointStruct(
                        id=job_id,
                        payload={"_id": job["_id"]},
                        sparse_vectors={
                            "bm25": sparse_embedding.as_object()
                        }
                    )
                ]
            )
        except Exception as e2:
            print(f"Alternative method also failed: {e2}")

# Create query embeddings for both models
dense_query_vectors = []
sparse_query_vectors = []

for query in job_queries:
    # Create dense embedding with your fine-tuned model
    dense_query_embedding = list(job_model.encode([query["text"]]))[0]
    dense_query_vectors.append(dense_query_embedding)
    
    # Create sparse embedding with BM25
    sparse_query_embedding = list(bm25_embedding_model.query_embed([query["text"]]))[0]
    sparse_query_vectors.append(sparse_query_embedding)

# Function to perform hybrid search
def job_hybrid_search(
    client, 
    query_id,
    dense_vector, 
    sparse_vector,
    dense_weight=0.7, 
    sparse_weight=0.3,
    limit=10
):
    # Normalize weights
    total = dense_weight + sparse_weight
    dense_weight /= total
    sparse_weight /= total
    search_limit = limit * 3
    
    # Query dense collection
    dense_results = client.query_points(
        "software_jobs_dense",
        query=dense_vector,
        using="job_embeddings",
        with_payload=False,
        limit=search_limit,
    )
    
    # Query sparse collection using the correct syntax
    sparse_results = client.query_points(
        "software_jobs_sparse",
        query=models.SparseVector(
            indices=sparse_vector.as_object()["indices"].tolist(),
            values=sparse_vector.as_object()["values"].tolist(),
        ),
        using="bm25",  # The name of your sparse vector field
        with_payload=False,
        limit=search_limit,
    )
    
    # Combine and normalize scores
    combined_scores = {}
    
    # Process dense results
    dense_scores = {str(point.id): point.score for point in dense_results.points}
    if dense_scores:
        max_dense = max(dense_scores.values()) if dense_scores else 0
        min_dense = min(dense_scores.values()) if dense_scores else 0
        range_dense = max_dense - min_dense if max_dense > min_dense else 1.0
        
        for doc_id, score in dense_scores.items():
            norm_score = (score - min_dense) / range_dense if range_dense > 0 else score
            combined_scores[doc_id] = dense_weight * norm_score
    
    # Process sparse results
    sparse_scores = {str(point.id): point.score for point in sparse_results.points}
    if sparse_scores:
        max_sparse = max(sparse_scores.values()) if sparse_scores else 0
        min_sparse = min(sparse_scores.values()) if sparse_scores else 0
        range_sparse = max_sparse - min_sparse if max_sparse > min_sparse else 1.0
        
        for doc_id, score in sparse_scores.items():
            norm_score = (score - min_sparse) / range_sparse if range_sparse > 0 else score
            if doc_id in combined_scores:
                combined_scores[doc_id] += sparse_weight * norm_score
            else:
                combined_scores[doc_id] = sparse_weight * norm_score
    
    # Sort by score and limit results
    sorted_scores = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)[:limit]
    return dict(sorted_scores)

# Evaluate dense-only approach
dense_run_dic = {}
for idx, query in enumerate(job_queries):
    query_id = query["_id"]
    question_vector = dense_query_vectors[idx]
    
    results = client.query_points(
        "software_jobs_dense",
        query=question_vector,
        using="job_embeddings",
        with_payload=False,
        limit=10,
    )
    
    dense_run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }

# Evaluate hybrid approach with 0.8 dense, 0.2 sparse weights
hybrid_run_dic = {}
for idx, query in enumerate(job_queries):
    query_id = query["_id"]
    question_dense_vector = dense_query_vectors[idx]
    question_sparse_vector = sparse_query_vectors[idx]
    
    results = job_hybrid_search(
        client=client,
        query_id=query_id,
        dense_vector=question_dense_vector,
        sparse_vector=question_sparse_vector,
        dense_weight=0.8,  # Updated weights as requested
        sparse_weight=0.2,
        limit=10
    )
    
    hybrid_run_dic[query_id] = results

# Evaluate both approaches
dense_eval = simple_evaluate(
    custom_qrels.qrels,
    dense_run_dic,
    metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
)

hybrid_eval = simple_evaluate(
    custom_qrels.qrels,
    hybrid_run_dic,
    metrics=["precision@10", "recall@10", "mrr@10", "dcg@10"]
)

# Print only the two requested results
# print("Fine-tuned model results:", dense_eval)
# print("Hybrid (0.8 dense, 0.2 sparse) results:", hybrid_eval)

Error uploading sparse vector: 1 validation error for PointStruct
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'indices': [413...8, 1.5233973492020558]}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
Alternative method also failed: 2 validation errors for PointStruct
vector
  Field required [type=missing, input_value={'id': 1, 'payload': {'_i...4657082, 1961226291])}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'values': array...34657082, 1961226291])}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden
Error uploading sparse vector: 1 validation error for PointStruct
sparse_vectors
  Extra inputs are not permitted [type=extra_forbidden, input_value={'bm25': {'indices': [169...16, 1.549807

In [20]:
import pandas as pd 
import numpy as np

# Create dictionaries for each model's results
fine_tuned_results = {
    'precision@10': 0.6920000000000001, 
    'recall@10': 0.2941698942765772, 
    'mrr@10': 0.884, 
    'dcg@10': np.float64(7.432169888932931)
}

hybrid_results = {
    'precision@10': 0.6920000000000001, 
    'recall@10': 0.2941698942765772, 
    'mrr@10': 0.884, 
    'dcg@10': np.float64(7.432169888932931)
}

# Create a DataFrame to compare models
comparison_df = pd.DataFrame({
    'Fine-tuned Model': fine_tuned_results,
    'Hybrid (0.8 dense, 0.2 sparse)': hybrid_results
}).T

# Format the DataFrame for better readability
pd.set_option('display.float_format', '{:.4f}'.format)

# Display the DataFrame
comparison_df

,precision@10,recall@10,mrr@10,dcg@10
Fine-tuned Model,0.6920,0.2942,0.8840,7.4322
"Hybrid (0.8 dense, 0.2 sparse)",0.6920,0.2942,0.8840,7.4322
